In [1]:
import sys
from pathlib import Path
import warnings
from sklearn.exceptions import ConvergenceWarning


ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT set to:", ROOT)


from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from utils import discretize_preprocess
from katabatic.models.codi.models import CODI


warnings.filterwarnings("ignore", category=ConvergenceWarning)          # sklearn MLP
warnings.filterwarnings("ignore", message="Parameters: {")              # XGBoost unused params
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")


dataset_name = "magic"   

dataset_path = ROOT / "raw_data" / f"{dataset_name}.csv"
output_path = ROOT / "discretized_data" / f"{dataset_name}.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

# Preprocess (same style you used for nursery)
print(f"Discretizing {dataset_path} -> {output_path}")
discretize_preprocess(str(dataset_path), str(output_path))

# Paths for pipeline
input_csv     = str(output_path)
output_dir    = str(ROOT / "sample_data" / dataset_name)
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / dataset_name / "codi")

print("input_csv    :", input_csv)
print("output_dir   :", output_dir)
print("real_test_dir:", real_test_dir)
print("synthetic_dir:", synthetic_dir)

class CoDiCar(CODI):
    def __init__(self):
        super().__init__(
            # Diffusion hyperparameters
            n_steps=50,        # you can increase (e.g. 100) if you want stronger sampling
            beta_1=1e-5,
            beta_T=0.02,

            # Network architecture
            encoder_dim_con=(64, 128, 256),
            encoder_dim_dis=(64, 128, 256),
            nf_con=16,
            nf_dis=64,
            activation="relu",

            # Training hyperparameters
            epochs=30,         # bump up (e.g. 50) if you want more training for car (1781 rows)
            batch_size=512,
            lr_con=2e-3,
            lr_dis=2e-3,
            grad_clip=1.0,

            # Contrastive learning weights
            lambda_con=0.2,
            lambda_dis=0.2,

            # Misc
            random_state=42,
            device=None,       # auto: cuda if available, otherwise cpu
        )

pipeline = TrainTestSplitPipeline(
    model=lambda: CoDiCar()
)

result = pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
)

print("\nPipeline result (should include TSTR metrics + result path):")
print(result)


ROOT set to: C:\Users\Prabu\Downloads\Katabatic
Discretizing C:\Users\Prabu\Downloads\Katabatic\raw_data\magic.csv -> C:\Users\Prabu\Downloads\Katabatic\discretized_data\magic.csv
Preprocessing: C:\Users\Prabu\Downloads\Katabatic\raw_data\magic.csv
Saved preprocessed discrete dataset to: C:\Users\Prabu\Downloads\Katabatic\discretized_data\magic.csv
input_csv    : C:\Users\Prabu\Downloads\Katabatic\discretized_data\magic.csv
output_dir   : C:\Users\Prabu\Downloads\Katabatic\sample_data\magic
real_test_dir: C:\Users\Prabu\Downloads\Katabatic\sample_data\magic
synthetic_dir: C:\Users\Prabu\Downloads\Katabatic\synthetic\magic\codi
Loaded data with shape: (19020, 11)


INFO:katabatic.models.codi.models:================================================================================
INFO:katabatic.models.codi.models:Training CoDi Model
INFO:katabatic.models.codi.models:================================================================================


Saved train/test full data
Train size: (15216, 11), Test size: (3804, 11)
Train label distribution:
 class
0    0.648396
1    0.351604
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.648265
1    0.351735
Name: proportion, dtype: float64
Saved X/y split
Training shape: (15216, 10) (15216,)
Test shape: (3804, 10) (3804,)


INFO:katabatic.models.codi.models:Loaded training data: (15216, 11)
INFO:katabatic.models.codi.models:Schema: 0 continuous, 11 categorical columns
INFO:katabatic.models.codi.models:Building models: con_dim=1, cat_dim=77
INFO:katabatic.models.codi.models:Continuous model params: 166,479
INFO:katabatic.models.codi.models:Discrete model params: 366,425
INFO:katabatic.models.codi.models:
Training for 30 epochs...
INFO:katabatic.models.codi.models:Epoch 1/30: loss_con=0.0000, loss_dis=88.8104
INFO:katabatic.models.codi.models:Epoch 5/30: loss_con=0.0000, loss_dis=85.9038
INFO:katabatic.models.codi.models:Epoch 10/30: loss_con=0.0000, loss_dis=85.5704
INFO:katabatic.models.codi.models:Epoch 15/30: loss_con=0.0000, loss_dis=85.4645
INFO:katabatic.models.codi.models:Epoch 20/30: loss_con=0.0000, loss_dis=85.4446
INFO:katabatic.models.codi.models:Epoch 25/30: loss_con=0.0000, loss_dis=85.3393
INFO:katabatic.models.codi.models:Epoch 30/30: loss_con=0.0000, loss_dis=85.2369
INFO:katabatic.models.


Results saved to: Results\magic\codi_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.7258
F1 Score: 0.6951
AUC: 0.7173

MLP:
Accuracy: 0.6391
F1 Score: 0.6405
AUC: 0.6560

RF:
Accuracy: 0.7369
F1 Score: 0.7412
AUC: 0.8040

XGBoost:
Accuracy: 0.6769
F1 Score: 0.6831
AUC: 0.8025

Pipeline result (should include TSTR metrics + result path):
Train test split pipeline executed successfully.


In [5]:
import os
from pathlib import Path

results_root = Path(r"C:\Users\Prabu\Downloads\Katabatic\Results")

print("Folders inside Results:")
print(os.listdir(results_root))



Folders inside Results:
['car', 'synthetic']


In [6]:
import os
from pathlib import Path

root = Path(r"C:\Users\Prabu\Downloads\Katabatic")

print("Searching for any file named 'codi_tstr.csv' in your entire Katabatic folder...\n")

for path, dirs, files in os.walk(root):
    for file in files:
        if file == "codi_tstr.csv":
            print("FOUND:", os.path.join(path, file))


Searching for any file named 'codi_tstr.csv' in your entire Katabatic folder...

FOUND: C:\Users\Prabu\Downloads\Katabatic\examples\Results\car\codi_tstr.csv
FOUND: C:\Users\Prabu\Downloads\Katabatic\Results\car\codi_tstr.csv


In [7]:
import sys
from pathlib import Path
import warnings
from sklearn.exceptions import ConvergenceWarning

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

# Try XGBoost if installed
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠️ xgboost not installed – XGBoost model will be skipped.")


# ----------------- ROOT + paths -----------------
ROOT = Path(r"C:\Users\Prabu\Downloads\Katabatic").resolve()
dataset_name = "magic"

real_dir = ROOT / "sample_data" / dataset_name          # x_test, y_test
synth_dir = ROOT / "synthetic" / dataset_name / "codi"  # x_synth, y_synth

print("Real dir   :", real_dir)
print("Synth dir  :", synth_dir)

# ----------------- Load data -----------------
x_test_real_path = real_dir / "x_test.csv"
y_test_real_path = real_dir / "y_test.csv"
x_synth_path     = synth_dir / "x_synth.csv"
y_synth_path     = synth_dir / "y_synth.csv"

print("\nChecking files exist:")
print("x_test_real:", x_test_real_path.exists(), x_test_real_path)
print("y_test_real:", y_test_real_path.exists(), y_test_real_path)
print("x_synth    :", x_synth_path.exists(), x_synth_path)
print("y_synth    :", y_synth_path.exists(), y_synth_path)

# If any are missing, stop early
for p in [x_test_real_path, y_test_real_path, x_synth_path, y_synth_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")

X_test_real = pd.read_csv(x_test_real_path)
y_test_real = pd.read_csv(y_test_real_path).iloc[:, 0]

X_synth = pd.read_csv(x_synth_path)
y_synth = pd.read_csv(y_synth_path).iloc[:, 0]

print("\nShapes:")
print("Real test   :", X_test_real.shape, y_test_real.shape)
print("Synthetic   :", X_synth.shape, y_synth.shape)

# ----------------- Helper: run one classifier -----------------
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="Parameters: {")
warnings.filterwarnings("ignore", category=UserWarning, module="xgboost")

def run_tstr_classifier(model_name: str):
    scaler = StandardScaler()
    X_synth_scaled = scaler.fit_transform(X_synth)
    X_test_scaled  = scaler.transform(X_test_real)

    if model_name == "LR":
        clf = LogisticRegression(max_iter=1000, n_jobs=-1)
    elif model_name == "MLP":
        clf = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42)
    elif model_name == "RF":
        clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    elif model_name == "XGBoost":
        if not HAS_XGB:
            print("⏭ Skipping XGBoost – xgboost not installed.")
            return None
        clf = XGBClassifier(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.1,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            n_jobs=-1,
            random_state=42,
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")

    clf.fit(X_synth_scaled, y_synth)
    y_pred = clf.predict(X_test_scaled)

    acc = accuracy_score(y_test_real, y_pred)
    f1  = f1_score(y_test_real, y_pred, average="weighted")
    return acc, f1

# ----------------- Run all models + save to Downloads -----------------
rows = []
for model_name in ["LR", "MLP", "RF", "XGBoost"]:
    print(f"\nRunning MAGIC CoDi TSTR for {model_name}...")
    res = run_tstr_classifier(model_name)
    if res is None:
        continue
    acc, f1 = res
    print(f"{model_name} -> Accuracy = {acc:.4f}, F1 = {f1:.4f}")
    rows.append({
        "Model": model_name,
        "Accuracy": round(acc, 4),
        "F1": round(f1, 4),
    })

df_results = pd.DataFrame(rows, columns=["Model", "Accuracy", "F1"])
display(df_results)

# Save directly to Downloads
downloads_csv = Path(r"C:\Users\Prabu\Downloads") / "codi_magic_tstr.csv"
df_results.to_csv(downloads_csv, index=False)

print("\n✅ MAGIC CoDi TSTR results saved to:")
print(downloads_csv)


Real dir   : C:\Users\Prabu\Downloads\Katabatic\sample_data\magic
Synth dir  : C:\Users\Prabu\Downloads\Katabatic\synthetic\magic\codi

Checking files exist:
x_test_real: True C:\Users\Prabu\Downloads\Katabatic\sample_data\magic\x_test.csv
y_test_real: True C:\Users\Prabu\Downloads\Katabatic\sample_data\magic\y_test.csv
x_synth    : True C:\Users\Prabu\Downloads\Katabatic\synthetic\magic\codi\x_synth.csv
y_synth    : True C:\Users\Prabu\Downloads\Katabatic\synthetic\magic\codi\y_synth.csv

Shapes:
Real test   : (3804, 10) (3804,)
Synthetic   : (15226, 10) (15226,)

Running MAGIC CoDi TSTR for LR...
LR -> Accuracy = 0.7258, F1 = 0.6951

Running MAGIC CoDi TSTR for MLP...
MLP -> Accuracy = 0.6391, F1 = 0.6405

Running MAGIC CoDi TSTR for RF...
RF -> Accuracy = 0.7363, F1 = 0.7417

Running MAGIC CoDi TSTR for XGBoost...
XGBoost -> Accuracy = 0.7479, F1 = 0.7529


,Model,Accuracy,F1
0,LR,0.7258,0.6951
1,MLP,0.6391,0.6405
2,RF,0.7363,0.7417
3,XGBoost,0.7479,0.7529



✅ MAGIC CoDi TSTR results saved to:
C:\Users\Prabu\Downloads\codi_magic_tstr.csv
